<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## KKT: When a Bound Stops Improvement

**See why the best allowed clock setting can stop at a boundary, even while the error is still decreasing.**

Keep the same clock observations, squared-error objective, and exact calibration at one hour. Add an upper limit on the initial correction. Two slider examples show how that limit changes the allowed settings and the best choice.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · A lower error is useful only when the setting is allowed

Choose an initial correction $q$ (min) and a rate correction $r$ (min/h). A negative rate means that the correction decreases over time. The four fixed observations are:

| Time $t_i$ (h) | Recorded error $d_i$ (min) |
|:---|:---|
| 0 | $-2$ |
| 1 | $-1$ |
| 2 | $-1$ |
| 3 | $0$ |

For reading $i=1,\ldots,4$, the remaining error is $y_i$. Squaring and adding the errors gives the objective $\phi$:

$$
y_i=d_i+q+rt_i,\qquad \phi(q,r)=\sum_{i=1}^4 y_i^2.
$$

Calibration makes the one-hour error zero: $-1+q+r=0$. Thus **choosing $q$ also fixes $r=1-q$**. These equations use numerical values in the stated units. Substitution leaves a one-variable score, called $p$:

$$
p(q)=\phi(q,1-q)=0.5+6(q-1.5)^2.
$$

The square is smallest at $q=1.5$, giving $r=-0.5$ and score $0.5$ min². Now impose the upper limit $q\le q_{\max}$ with $q_{\max}=1$ min. The lowest point of the curve is no longer allowed.

At $q=1$, calibration gives $r=0$. The errors are $(-1,0,0,1)$ min, so the score is $1+0+0+1=2$ min². Increasing $q$ would reduce the error, but it would break the limit. The best allowed point therefore sits at the end of the teal curve.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/clock_bound_concept.svg" alt="The calibrated clock score decreases toward correction 1.5, but the upper limit at 1 blocks that improvement. Teal is allowed and gray is forbidden." width="512" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Move the setting: lower error can still be rejected

The first slider changes only the candidate correction $q$; the rate follows $r=1-q$. The limit stays at $q_{\max}=1$. Orange marks an allowed candidate; gray marks a rejected one.

At $q=0.75$, the score is $3.875$ min² and increasing $q$ helps. At $q=1$, the score reaches the best allowed value, $2$ min². At $q=1.5$, the score is lower, but the candidate is rejected. **Feasibility comes before comparing scores.**

The code below calculates the same four errors and checks the requirements. The plotting range is a viewing window, not an additional constraint.


In [ ]:
import sys
import numpy as np
import matplotlib

if sys.platform != "emscripten":
    try:
        matplotlib.use("widget", force=True)
    except (RuntimeError, ValueError):
        from matplotlib.backends import backend_registry
        backend_registry._clear()
        matplotlib.use("widget", force=True)
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

BLUE, TEAL, ORANGE, GRAY = "#2878b5", "#168578", "#e78b24", "#707070"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11})

# Fixed observations and the retained calibration requirement.
observation_times = np.array([0., 1., 2., 3.])  # h
observed_errors = np.array([-2., -1., -1., 0.])  # min
calibration_target = 1.0  # min, at one hour
check_tolerance = 1e-9  # Numerical comparison tolerance, not a physical limit.


def evaluate_clock(correction, upper_limit):
    rate = calibration_target - correction
    # System response and performance.
    errors = observed_errors + correction + rate * observation_times
    objective = float(errors @ errors)
    gradient = 2 * np.array([errors.sum(), observation_times @ errors])
    # Requirement checks, separate from the score.
    equality_residual = correction + rate - calibration_target
    bound_residual = correction - upper_limit
    feasible = (abs(equality_residual) <= check_tolerance
                and bound_residual <= check_tolerance)
    return {"decision": np.array([correction, rate]), "errors": errors,
            "objective": objective, "gradient": gradient,
            "slope": float(gradient[0] - gradient[1]),
            "equality_residual": equality_residual,
            "bound_residual": bound_residual, "slack": upper_limit - correction,
            "feasible": feasible,
            "active": abs(bound_residual) <= check_tolerance}


def solve_bound(upper_limit):
    # Exact minimum of 0.5 + 6*(q - 1.5)**2 over q <= upper_limit.
    correction = min(1.5, upper_limit)
    result = evaluate_clock(correction, upper_limit)
    bound_multiplier = max(0.0, -result["slope"])
    equality_multiplier = -result["gradient"][1]
    result.update({"bound_multiplier": bound_multiplier,
                   "equality_multiplier": equality_multiplier,
                   "stationarity": result["gradient"] + equality_multiplier
                       + np.array([bound_multiplier, 0.]),
                   "complementarity": bound_multiplier * result["bound_residual"]})
    return result


def make_score_plot(title, height=5.0):
    figure, axis = plt.subplots(figsize=(5.6, height))
    figure.subplots_adjust(left=0.15, right=0.96, top=0.90, bottom=0.46)
    corrections = np.linspace(0.45, 2.3, 300)
    scores = [evaluate_clock(q, 1.0)["objective"] for q in corrections]
    axis.plot(corrections, scores, color=GRAY, linewidth=2)
    allowed_curve, = axis.plot([], [], color=TEAL, linewidth=3)
    boundary = axis.axvline(1, color=TEAL, linestyle="--", linewidth=1.5)
    axis.plot(1.5, 0.5, "x", color=GRAY, markersize=8)
    candidate, = axis.plot([], [], "o", color=ORANGE, markersize=8, zorder=4)
    axis.set(xlabel="Correction q (min)", ylabel="Squared error (min²)",
             title=title, xlim=(0.45, 2.3), ylim=(0, 7),
             xticks=[0.5, 1, 1.5, 2], yticks=[0, 2, 4, 6])
    axis.grid(alpha=0.25)
    axis.text(0.97, 0.94, "Teal: allowed    Gray: forbidden", transform=axis.transAxes,
              va="top", ha="right", fontsize=10,
              bbox={"facecolor": "white", "edgecolor": "none", "pad": 2})
    return figure, axis, allowed_curve, boundary, candidate


def draw_allowed_curve(allowed_curve, boundary, upper_limit):
    corrections = np.linspace(0.45, upper_limit, 160)
    allowed_curve.set_data(corrections, 0.5 + 6 * (corrections - 1.5)**2)
    boundary.set_xdata([upper_limit, upper_limit])


In [ ]:
candidate_figure, candidate_axis, candidate_curve, fixed_boundary, candidate_point = (
    make_score_plot("A lower score can violate the limit"))
draw_allowed_curve(candidate_curve, fixed_boundary, 1.0)
candidate_status = candidate_figure.text(0.15, 0.30, "", va="top", linespacing=1.4)
correction_slider = Slider(candidate_figure.add_axes([0.29, 0.065, 0.53, 0.035]),
                           "q (min)", 0.5, 2.25, valinit=0.75, valstep=0.05,
                           valfmt="%1.2f", color=BLUE)


def update_candidate(_):
    result = evaluate_clock(correction_slider.val, upper_limit=1.0)
    correction, rate = result["decision"]
    color = ORANGE if result["feasible"] else GRAY
    candidate_point.set_data([correction], [result["objective"]])
    candidate_point.set_color(color)
    if not result["feasible"]:
        explanation = "REJECTED: q exceeds the fixed limit of 1 min."
    elif result["active"]:
        explanation = "BEST ALLOWED: moving right breaks the limit."
    else:
        explanation = "ALLOWED: moving right can still lower the error."
    candidate_status.set_text(
        f"q = {correction:.2f} min; r = {rate:.2f} min/h\n"
        f"Score = {result['objective']:.3f} min²\n{explanation}")
    candidate_figure._result = result
    candidate_figure.canvas.draw_idle()


correction_slider.on_changed(update_candidate)
candidate_figure._sliders = [correction_slider]
update_candidate(None)
plt.show()


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Move the limit: the best setting follows, then stops

The second slider changes only the requirement $q_{\max}$. For each limit, the orange point is recalculated as the **exact best allowed setting**:

$$
q^\star=\min(1.5,q_{\max}),\qquad r^\star=1-q^\star.
$$

The star means optimal. Before the limit reaches $1.5$, the best point follows the boundary. Beyond $1.5$, it stays at the bottom of the curve. The observations, calibration, and objective do not change.

A bound is **active** when the selected point touches it; otherwise it is **inactive**. Slack is the unused room, $q_{\max}-q^\star$.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/bound_activity_and_multiplier.svg" alt="At limits 1, 1.5, and 2, the best corrections are 1, 1.5, and 1.5. The bound is active in the first two cases; the multipliers are 6, 0, and 0." width="512" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The displayed multiplier $\mu$ measures how much the bound blocks improvement at the optimum. It is calculated from the curve's slope, not chosen as a score weight. Here

$$
p'(q)=12(q-1.5),\qquad \mu=-p'(q^\star).
$$

| Limit $q_{\max}$ (min) | Best $q^\star$ (min) | Score (min²) | Slack (min) | $\mu$ | What stops improvement |
|:---|:---|:---|:---|:---|:---|
| 1 | 1 | 2 | 0 | 6 | The bound blocks movement toward lower error. |
| 1.5 | 1.5 | 0.5 | 0 | 0 | The boundary is exactly at the curve's bottom. |
| 2 | 1.5 | 0.5 | 0.5 | 0 | The curve's bottom is inside the allowed region. |

At the tight limit, the slope is $-6$ and $\mu=6$ balances it. At a loose limit, both are zero. **Touching the boundary does not always mean a positive multiplier.** The readout shows that distinction as the limit passes $1.5$.


In [ ]:
bound_figure, bound_axis, bound_curve, moving_boundary, optimum_point = (
    make_score_plot("The best setting follows the bound, then stops", height=5.4))
bound_status = bound_figure.text(0.15, 0.31, "", va="top", linespacing=1.4)
limit_slider = Slider(bound_figure.add_axes([0.29, 0.055, 0.53, 0.035]),
                      "Limit (min)", 0.5, 2.25, valinit=1.0, valstep=0.05,
                      valfmt="%1.2f", color=TEAL)


def update_bound(_):
    upper_limit = limit_slider.val
    result = solve_bound(upper_limit)
    correction, rate = result["decision"]
    multiplier = result["bound_multiplier"]
    draw_allowed_curve(bound_curve, moving_boundary, upper_limit)
    optimum_point.set_data([correction], [result["objective"]])
    if multiplier > check_tolerance:
        explanation = "ACTIVE: the bound blocks lower error."
    elif result["active"]:
        explanation = "ACTIVE, mu = 0: already at the curve's bottom."
    else:
        explanation = "INACTIVE: room remains beyond the best setting."
    bound_status.set_text(
        f"Best q = {correction:.2f}; r = {rate:.2f}; score = {result['objective']:.3f}\n"
        f"Slack = {result['slack']:.2f} min; mu = {multiplier:.2f}\n"
        f"Slope + mu = {result['slope']:.2f} + {multiplier:.2f} = 0\n"
        f"{explanation}")
    bound_figure._result = result
    bound_figure.canvas.draw_idle()


limit_slider.on_changed(update_bound)
bound_figure._sliders = [limit_slider]
update_bound(None)
plt.show()


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · KKT names the checks behind this picture

The **Karush–Kuhn–Tucker (KKT) conditions** describe a feasible candidate whose objective and constraint slopes balance. Along our calibrated curve, they become four short checks:

| Check | Equation | Meaning in the picture |
|:---|:---|:---|
| Feasibility | $q\le q_{\max}$ | The point is allowed; $r=1-q$ already enforces calibration. |
| Multiplier sign | $\mu\ge0$ | An upper bound can block movement to the right, not to the left. |
| Complementarity | $\mu(q-q_{\max})=0$ | If room remains before the bound, its multiplier must be zero. |
| Stationarity | $p'(q)+\mu=0$ | A downhill slope is balanced by the bound, or both terms are zero. |

At $q_{\max}=1$, $q=1$ and $\mu=6$ pass all four checks: $1\le1$, $6\ge0$, $6(1-1)=0$, and $-6+6=0$.

The sign condition also rejects a misleading boundary point. With limit $2$, choosing $q=2$ touches the boundary but gives slope $p'(2)=6$. Balancing it would require $\mu=-6$, which is forbidden. Moving left is allowed and lowers the error, so this boundary point cannot be optimal.

For the original two clock settings, name the residuals $h=q+r-1$ and $g=q-q_{\max}$. The Lagrangian includes the equality multiplier $\nu$ as well:

$$
L=\phi(q,r)+\nu h+\mu g.
$$

Feasibility means $h=0$ and $g\le0$. The conditions $\mu\ge0$ and $\mu g=0$ are unchanged; $\nu$ can have either sign. Stationarity now balances both setting slopes:

$$
\frac{\partial\phi}{\partial q}+\nu+\mu=0,
\qquad
\frac{\partial\phi}{\partial r}+\nu=0.
$$

At $(q,r)=(1,0)$, the objective partial derivatives are $0$ and $6$. Choosing $\nu=-6$ and $\mu=6$ gives $0-6+6=0$ and $6-6=0$. Subtracting the two equations recovers the one-variable balance $p'(q)+\mu=0$. The slider picture therefore retains both the equality and the inequality.

For this bowl-shaped quadratic with linear constraints, these checks certify the unique global minimum. More generally, KKT is necessary at a smooth local minimum under a constraint regularity condition, such as independent equality and active-inequality gradients; it does not always certify a minimum. KKT gives conditions for checking a candidate, not a search algorithm.
